# RegUQ Tutorial: Getting Started with Regression Uncertainty Quantification

This notebook demonstrates how to use the RegUQ package for uncertainty quantification in regression tasks.

## What You'll Learn

1. Installing and importing RegUQ
2. Loading data and preparing for analysis
3. Running different uncertainty quantification phases
4. Interpreting results and generating reports
5. Using configuration files for reproducible workflows

## Prerequisites

- Python 3.10+
- Basic understanding of regression and machine learning
- Familiarity with pandas and scikit-learn

## 1. Installation

First, let's install the RegUQ package from GitHub.

In [ ]:
# Install RegUQ
!pip install -q "git+https://github.com/DaneshSelwal/reguq.git"

# If running in Google Colab, use the bootstrap function (see Colab section below)

## 2. Import Libraries and Generate Sample Data

For this tutorial, we'll create synthetic data. In practice, you'd load your own dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic regression data
n_samples = 500
n_features = 5

# Create features
X = np.random.randn(n_samples, n_features)

# Create target with some non-linear relationship and noise
y = (
    3 * X[:, 0] +
    2 * X[:, 1] ** 2 +
    np.sin(X[:, 2]) * 5 +
    X[:, 3] * X[:, 4] +
    np.random.randn(n_samples) * 2  # Add noise
)

# Create DataFrames
feature_names = [f'feature_{i}' for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# Split into train and test
train_size = int(0.8 * n_samples)
train_df = df[:train_size].copy()
test_df = df[train_size:].copy()

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nFirst few rows of training data:")
train_df.head()

## 3. Quantile Regression: Prediction Intervals

Quantile regression predicts conditional quantiles to create prediction intervals.

In [ ]:
from reguq import run_quantile

# Run quantile regression
quantile_result = run_quantile(
    data={
        "train": train_df,
        "test": test_df
    },
    target_col="target",
    models=["lightgbm", "xgboost"],
    params_source={"mode": "defaults"},
    output_config={
        "output_dir": "./outputs/tutorial_quantile",
        "export_excel": True,
        "export_plots": True,
        "show_inline_plots": True,
        "chart_detail_level": "detailed"
    }
)

print("\n=== Quantile Regression Results ===")
for model_name, metrics in quantile_result.metrics.items():
    print(f"\n{model_name.upper()}:")
    print(f"  RMSE: {metrics['rmse']:.4f}")
    print(f"  MAE: {metrics['mae']:.4f}")
    print(f"  Coverage (90% interval): {metrics['coverage']:.2%}")
    print(f"  Mean Interval Width: {metrics['mean_interval_width']:.4f}")

### Interpreting Quantile Results

- **RMSE/MAE**: Measure point prediction accuracy
- **Coverage**: Percentage of actual values within the prediction interval
  - Should be close to 90% for 90% intervals
  - Higher coverage = more conservative intervals
- **Mean Interval Width**: Average width of prediction intervals
  - Narrower is better (if coverage is maintained)
  - Trade-off between coverage and width

## 4. Probabilistic Regression: Full Distributions

Probabilistic models predict the full conditional probability distribution.

In [ ]:
from reguq import run_probabilistic

# Run probabilistic regression
prob_result = run_probabilistic(
    data={
        "train": train_df,
        "test": test_df
    },
    target_col="target",
    models=["ngboost"],  # NGBoost supports probabilistic predictions
    params_source={"mode": "defaults"},
    output_config={
        "output_dir": "./outputs/tutorial_probabilistic",
        "export_excel": True,
        "export_plots": True,
        "show_inline_plots": True
    }
)

print("\n=== Probabilistic Regression Results ===")
for model_name, metrics in prob_result.metrics.items():
    print(f"\n{model_name.upper()}:")
    print(f"  RMSE: {metrics['rmse']:.4f}")
    print(f"  NLL (Negative Log-Likelihood): {metrics['nll']:.4f}")
    print(f"  CRPS (Continuous Ranked Probability Score): {metrics['crps']:.4f}")
    if 'calibration_error' in metrics:
        print(f"  Calibration Error: {metrics['calibration_error']:.4f}")

### Interpreting Probabilistic Results

- **NLL (Negative Log-Likelihood)**: Measures how well the predicted distribution matches the actual data
  - Lower is better
  - Proper scoring rule (encourages honest predictions)
  
- **CRPS**: Continuous Ranked Probability Score
  - Generalization of MAE to probability distributions
  - Lower is better
  - Proper scoring rule
  
- **Calibration Error**: Measures if predicted uncertainties match observed frequencies
  - Lower is better
  - 0 = perfect calibration

## 5. Conformal Prediction: Distribution-Free Coverage

Conformal prediction provides valid coverage guarantees without distributional assumptions.

In [ ]:
from reguq import run_conformal_standard

# Run conformal prediction
conformal_result = run_conformal_standard(
    data={
        "train": train_df,
        "test": test_df
    },
    target_col="target",
    models=["lightgbm"],
    params_source={"mode": "defaults"},
    conformal_config={
        "alpha": 0.1,  # 90% coverage
        "methods": ["mapie"],
        "mapie_method": "plus"  # Jackknife+
    },
    output_config={
        "output_dir": "./outputs/tutorial_conformal",
        "export_excel": True,
        "export_plots": True,
        "show_inline_plots": True
    }
)

print("\n=== Conformal Prediction Results ===")
for model_name, metrics in conformal_result.metrics.items():
    print(f"\n{model_name.upper()}:")
    print(f"  Target Coverage: 90%")
    print(f"  Actual Coverage: {metrics['coverage']:.2%}")
    print(f"  Mean Interval Width: {metrics['mean_interval_width']:.4f}")
    print(f"  Coverage Guarantee: YES (by conformal theory)")

### Why Conformal Prediction?

- **Finite-sample guarantees**: Valid coverage for any sample size
- **Distribution-free**: No assumptions about data distribution
- **Model-agnostic**: Works with any base model
- **Theoretically grounded**: Coverage guarantee: P(Y ∈ [L, U]) ≥ 1 - α

## 6. Hyperparameter Tuning (Optional)

For better performance, tune hyperparameters using Optuna.

In [ ]:
from reguq import run_tuning

# Run hyperparameter tuning (this may take a few minutes)
tuning_result = run_tuning(
    data={
        "train": train_df,
        "test": test_df
    },
    target_col="target",
    models=["lightgbm"],
    tuning_config={
        "n_trials": 20,  # Number of Optuna trials (increase for better results)
        "cv": 3,  # Cross-validation folds
        "scoring": "neg_root_mean_squared_error",
        "random_state": 42
    },
    output_config={
        "output_dir": "./outputs/tutorial_tuning"
    }
)

print("\n=== Tuning Results ===")
for model_name, params in tuning_result.best_params.items():
    print(f"\n{model_name.upper()} Best Parameters:")
    for param, value in params.items():
        print(f"  {param}: {value}")
    print(f"  Best Score: {tuning_result.best_scores[model_name]:.4f}")

### Using Tuned Parameters

After tuning, use the optimized parameters:

In [ ]:
# Run quantile regression with tuned parameters
tuned_result = run_quantile(
    data={
        "train": train_df,
        "test": test_df
    },
    target_col="target",
    models=["lightgbm"],
    params_source={
        "mode": "load_only",  # Load tuned params
        "params_dir": "./outputs/tutorial_tuning"
    },
    output_config={
        "output_dir": "./outputs/tutorial_quantile_tuned",
        "export_excel": True
    }
)

print("\nTuned Model Performance:")
print(f"RMSE: {tuned_result.metrics['lightgbm']['rmse']:.4f}")
print(f"Coverage: {tuned_result.metrics['lightgbm']['coverage']:.2%}")

## 7. Complete Pipeline with Config File

For reproducible workflows, use a configuration file.

In [ ]:
import yaml

# Create a configuration dictionary
config = {
    "data": {
        "train": train_df.to_dict('list'),  # Convert to dict for YAML
        "test": test_df.to_dict('list'),
        "target_col": "target"
    },
    "models": ["lightgbm", "xgboost"],
    "phases": ["quantile", "probabilistic"],
    "params_source": {
        "mode": "defaults"
    },
    "output": {
        "output_dir": "./outputs/tutorial_pipeline",
        "export_excel": True,
        "export_plots": True,
        "chart_detail_level": "detailed"
    }
}

# Save config to file
with open("tutorial_config.yaml", "w") as f:
    yaml.dump(config, f)

print("Configuration saved to tutorial_config.yaml")

**Note**: When working with CSV files (recommended for production), your config would look like:

```yaml
data:
  train_path: ./data/train.csv
  test_path: ./data/test.csv
  target_col: target

models:
  - lightgbm
  - xgboost

phases:
  - quantile
  - probabilistic

output:
  output_dir: ./outputs/my_run
  export_excel: true
```

## 8. Accessing and Visualizing Results

Let's examine the outputs in detail.

In [ ]:
# Access predictions for a specific model
lightgbm_preds = quantile_result.predictions["lightgbm"]

print("Prediction DataFrame columns:")
print(lightgbm_preds.columns.tolist())
print("\nFirst few predictions:")
lightgbm_preds.head()

In [ ]:
# Custom visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Predictions vs Actuals with Intervals
ax1 = axes[0]
indices = range(len(lightgbm_preds))
ax1.scatter(lightgbm_preds['y_true'], lightgbm_preds['y_pred'], 
           alpha=0.5, label='Point Predictions')
ax1.plot([lightgbm_preds['y_true'].min(), lightgbm_preds['y_true'].max()],
        [lightgbm_preds['y_true'].min(), lightgbm_preds['y_true'].max()],
        'r--', label='Perfect Prediction')
ax1.set_xlabel('Actual Values')
ax1.set_ylabel('Predicted Values')
ax1.set_title('Predictions vs Actuals')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction Intervals
ax2 = axes[1]
sample_size = min(50, len(lightgbm_preds))  # Show first 50 samples
sample_preds = lightgbm_preds.head(sample_size)
x_range = range(sample_size)

ax2.plot(x_range, sample_preds['y_true'], 'ko-', label='Actual', linewidth=2, markersize=4)
ax2.plot(x_range, sample_preds['y_pred'], 'b^-', label='Predicted', linewidth=1.5, markersize=4)
ax2.fill_between(x_range, 
                 sample_preds['lower_bound'],
                 sample_preds['upper_bound'],
                 alpha=0.3, label='90% Interval')
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('Value')
ax2.set_title('Prediction Intervals (First 50 Samples)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate coverage manually
within_interval = (
    (lightgbm_preds['y_true'] >= lightgbm_preds['lower_bound']) &
    (lightgbm_preds['y_true'] <= lightgbm_preds['upper_bound'])
)
coverage = within_interval.mean()
print(f"\nEmpirical Coverage: {coverage:.2%}")
print(f"Points within interval: {within_interval.sum()}/{len(lightgbm_preds)}")

## 9. Google Colab Setup

If you're running this in Google Colab, use the bootstrap function:

In [ ]:
# ONLY RUN THIS IN GOOGLE COLAB
# Uncomment the following lines if you're in Colab:

# !pip install --upgrade --force-reinstall --no-cache-dir "git+https://github.com/DaneshSelwal/reguq.git"

# from reguq import bootstrap_colab_environment
# bootstrap_colab_environment(
#     repo_url="https://github.com/DaneshSelwal/reguq.git"
# )

# After runtime restart, continue with the tutorial from cell 2

## 10. Best Practices

### When to Use Each Method

1. **Quantile Regression**:
   - Fast and efficient
   - Good for simple prediction intervals
   - No distributional assumptions

2. **Probabilistic Models**:
   - When you need full distributions
   - For risk assessment and decision-making
   - When calibration is important

3. **Conformal Prediction**:
   - When coverage guarantees are critical
   - Distribution-free requirements
   - Compliance and regulatory needs

### Data Considerations

- **Time Series**: Set `shuffle=False` in split config
- **Small Datasets**: Use cross-validation in tuning
- **Large Datasets**: Reduce `n_trials` to save time
- **Imbalanced Targets**: Consider transformation or weighted metrics

### Performance Tips

1. Start with default parameters for exploration
2. Use tuning for production models
3. Export results for offline analysis
4. Monitor coverage metrics over time

## 11. Next Steps

Now that you've completed this tutorial, you can:

1. **Apply to Your Data**: Replace the synthetic data with your own CSV files
2. **Explore Advanced Features**: Check out `GUIDE.md` for more options
3. **Read Technical Details**: See `SKILL.md` for implementation details
4. **Try Different Models**: Experiment with catboost, ngboost, pgbm
5. **Optimize Further**: Run more tuning trials for better performance

## Resources

- **User Guide**: [GUIDE.md](../GUIDE.md)
- **Technical Documentation**: [SKILL.md](../SKILL.md)
- **GitHub Repository**: https://github.com/DaneshSelwal/reguq
- **Issues**: https://github.com/DaneshSelwal/reguq/issues

## Citation

If you use RegUQ in your research:

```
RegUQ: Regression Uncertainty Quantification Toolkit
Authors: Prakriti Bisht, Danesh Selwal
Under guidance of: Dr. Mahesh Pal
GitHub: https://github.com/DaneshSelwal/reguq
```

---

**Congratulations!** You've completed the RegUQ tutorial. You now know how to:

✅ Install and import RegUQ  
✅ Run quantile regression for prediction intervals  
✅ Use probabilistic models for full distributions  
✅ Apply conformal prediction for coverage guarantees  
✅ Tune hyperparameters for better performance  
✅ Interpret and visualize results  
✅ Use configuration files for reproducible workflows  

Happy uncertainty quantification! 🎉